# Composite MNIST multi-label classifier

This notebook clones the repository, downloads datasets from Google Drive, concatenates only their training splits, and evaluates validation/test splits separately.

In [1]:
!pip install -q gdown
!pip install -q matplotlib tqdm

## Configuration

In [7]:
REPO_URL = 'https://github.com/izahai/Multi-Label-Classification-with-MNIST-Digits.git'
BRANCH = 'main'
REPO_DIR = '/content/mnist-detector'

ORIGINAL_DATA_DIR = '/content/mnist-detector/data'
GEN_DATA_DIR = '/content/mnist-detector/data/gen_data'
OUTPUT_DIR = '/content/mnist-detector/results'

MODEL_SIZE = 'densenet_atn_head'  #  {'small', 'large', 'dense_net', 'densenet_atn_head'}
EPOCHS = 200
BATCH_SIZE = 64
NUM_WORKERS = 2
RESUME_CHECKPOINT = '/content/mnist-detector/results/last.pt'  # Optional path to last.pt on Drive

In [3]:
!git clone --branch "$BRANCH" "$REPO_URL" "$REPO_DIR";
!echo "Repository ready at $REPO_DIR"

Cloning into '/content/mnist-detector'...
remote: Enumerating objects: 126, done.
remote: Counting objects: 100% (126/126), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 126 (delta 62), reused 115 (delta 51), pack-reused 0 (from 0)
Receiving objects: 100% (126/126), 496.78 KiB | 852.00 KiB/s, done.
Resolving deltas: 100% (62/62), done.
Repository ready at /content/mnist-detector


## Download data and generated data

In [4]:
%cd /content/mnist-detector/data
# test.pt
!gdown --fuzzy "https://drive.google.com/file/d/192pw9kbsC7JOMwYUbWOyMppiJ85KDhU1/view?usp=sharing"
# train.pt
!gdown --fuzzy "https://drive.google.com/file/d/12XUEP0MAT_gxNOlOkjliR2ATRmZXNgY0/view?usp=sharing"
# val.pt
!gdown --fuzzy "https://drive.google.com/file/d/10Uk4BTATeUcswQrJXr2rwfL4exkvjSvt/view?usp=drive_link"

# self-generated train.pt
!mkdir gen_data
%cd gen_data
!gdown --fuzzy "https://drive.google.com/file/d/1DbI9glgKuXE0ZhfzHKmYR9Yp9ZBticDj/view?usp=drive_link"
%cd /content/mnist-detector

/content/mnist-detector/data
Downloading...
From (original): https://drive.google.com/uc?id=192pw9kbsC7JOMwYUbWOyMppiJ85KDhU1
From (redirected): https://drive.google.com/uc?id=192pw9kbsC7JOMwYUbWOyMppiJ85KDhU1&confirm=t&uuid=729e56c1-b779-45ca-a740-608cedbea1f5
To: /content/mnist-detector/data/test.pt
100% 44.2M/44.2M [00:00<00:00, 72.3MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=12XUEP0MAT_gxNOlOkjliR2ATRmZXNgY0
From (redirected): https://drive.google.com/uc?id=12XUEP0MAT_gxNOlOkjliR2ATRmZXNgY0&confirm=t&uuid=a87036bb-d855-4596-b294-10efcb69290c
To: /content/mnist-detector/data/train.pt
100% 221M/221M [00:01<00:00, 119MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=10Uk4BTATeUcswQrJXr2rwfL4exkvjSvt
From (redirected): https://drive.google.com/uc?id=10Uk4BTATeUcswQrJXr2rwfL4exkvjSvt&confirm=t&uuid=c5fafd83-1af4-4735-b3c4-f913bcfe6021
To: /content/mnist-detector/data/val.pt
100% 44.2M/44.2M [00:00<00:00, 152MB/s]
/content/mnist-detector/data/g

In [5]:
from pathlib import Path
import os
import subprocess

required = [
    Path(ORIGINAL_DATA_DIR) / f'{split}.pt'
    for split in ('train', 'val', 'test')
] + [Path(GEN_DATA_DIR) / 'train.pt']
missing = [str(path) for path in required if not path.is_file()]

if missing:
    raise FileNotFoundError('Missing dataset files:\n' + '\n'.join(missing))
print('Required original splits and generated data training split are available.')

Required original splits and generated data training split are available.


## Train

In [ ]:
!python -m src_model_cls.train \
    --original-data-dir "{ORIGINAL_DATA_DIR}" \
    --output-dir "{OUTPUT_DIR}" \
    --model-name "{MODEL_SIZE}" \
    --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE} \
    --num-workers {NUM_WORKERS} \
    --bbox-data-dir "{GEN_DATA_DIR}"
    # use this for resuming training from the lastest epoch
    #--resume {RESUME_CHECKPOINT}

Device: cuda
Loading and concatenating the two training sources...
Samples: train=550,000 (original=50,000, bbox=500,000) | validation=10,000
Model: densenet_atn_head | 6,965,396 total parameters | 6,965,396 trainable

Epoch 1/200
Train:  67% 5746/8594 [09:51<04:48,  9.88batch/s, binary=0.959, exact=0.749, loss=0.1008]

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(Path(OUTPUT_DIR) / 'training_curves.png')))

## Test

In [13]:
import json
import pandas as pd

evaluation_path = Path(OUTPUT_DIR) / "evaluation_metrics.json"
# CKPT = "/content/mnist-detector/results/top_checkpoints/epoch_001.pt"
BEST = "/content/mnist-detector/results/best.pt"

!python -m src_model_cls.evaluate \
  --checkpoint "{BEST}" \
  --original-data-dir "{ORIGINAL_DATA_DIR}" \
  --splits val test \
  --batch-size {BATCH_SIZE} \
  --num-workers {NUM_WORKERS} \
  --json-output "{evaluation_path}"

with evaluation_path.open() as file:
    results = json.load(file)
pd.DataFrame(results).T[['loss', 'exact_match', 'binary_match']]

Device: cuda
Model: densenet_atn_head
Checkpoint: /content/mnist-detector/results/best.pt
Presence threshold: 0.50
Original Val: 100% 157/157 [00:04<00:00, 35.56batch/s, loss=0.0189]
Original Val: loss=0.0189 exact=0.9694 binary=0.9960
Original Test: 100% 157/157 [00:03<00:00, 43.41batch/s, loss=0.0271]
Original Test: loss=0.0271 exact=0.9544 binary=0.9941
Saved metrics to /content/mnist-detector/results/evaluation_metrics.json


,loss,exact_match,binary_match
original_val,0.018894,0.9694,0.99595
original_test,0.027065,0.9544,0.99409
